In [23]:
# Environment variables
import os
from dotenv import load_dotenv

# OpenAI models & embeddings / OLLAMA
from langchain_ollama import ChatOllama
from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Document loading & text processing
from pypdf import PdfReader
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings & vector stores
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# LangChain tools
from langchain_core.tools import tool

# LangGraph core
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

# Messages
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage

# Typing
from typing import Annotated, TypedDict


In [4]:
load_dotenv()

REASONING_MODEL = os.getenv("MODEL_NAME")
OLLAMA_MODEL_DEEPSEEK = os.getenv("OLLAMA_MODEL_DEEPSEEK")
OPENAI_MODEL_GEMMA3 = os.getenv("OPENAI_MODEL_GEMMA3")

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "ollama")

In [5]:
print("GEMMA:", OPENAI_MODEL_GEMMA3)
print("OLLAMA URL:", OLLAMA_BASE_URL)

GEMMA: gemma3:4b
OLLAMA URL: http://localhost:11434


In [6]:
SYSTEM_PROMPT = SystemMessage(
    content=(
        "You are a helpful assistant."
    )
)

In [7]:
# Initialize Ollama LLM
llm_ollama = ChatOllama(
    model=OPENAI_MODEL_GEMMA3,
    base_url=OLLAMA_BASE_URL,
    temperature=0.8,
)

# Initialize ChatOpenAI (chat-based OpenAI models)
llm_chat_openai = ChatOpenAI(
    model=REASONING_MODEL,
    base_url=OPENAI_BASE_URL,
    temperature=0.8,
)

# Initialize OpenAI (non-chat / legacy or embeddings use)
llm_openai = OpenAI(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
)


In [8]:
pdf_path = "data_and_files/Investment.pdf"
reader = PdfReader(pdf_path)

# Fetch only the important metadata
pdf_info = reader.metadata

important_info = {
    "source": pdf_path,
    "title": pdf_info.title,
    "author": pdf_info.author,
    "subject": pdf_info.subject,
    "creator": pdf_info.creator,
    "producer": pdf_info.producer,
    "creation_date": pdf_info.creation_date,
    "modification_date": pdf_info.modification_date,
    "pages": len(reader.pages)
}

print(important_info)


{'source': 'data_and_files/Investment.pdf', 'title': None, 'author': 'net_pc', 'subject': None, 'creator': 'Microsoft® Word 2019', 'producer': 'Microsoft® Word 2019', 'creation_date': datetime.datetime(2024, 1, 13, 5, 50, 5, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200))), 'modification_date': datetime.datetime(2024, 1, 13, 5, 50, 5, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200))), 'pages': 65}


In [9]:
# Load PDF and split into pages
data = "data_and_files\\Investment.pdf"
loader = PyPDFLoader(data)
pages = loader.load_and_split()

In [ ]:
# pages

In [10]:
pages[0]

Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2024-01-13T05:50:05+02:00', 'author': 'net_pc', 'moddate': '2024-01-13T05:50:05+02:00', 'source': 'data_and_files\\Investment.pdf', 'total_pages': 65, 'page': 0, 'page_label': '1'}, page_content='1 | Page \nProf. Dr. Turgut Tursoy Investment \n \n \n \n \n \n \n \n \nInvestment \nProf. Dr. Turgut Tursoy \n \n \nIntro \n \nStart to understand that the in and waste would be \nfinding them together to create the new initiatives. It \nis the investment in life to understand a life that goes \nwith the extra efforts to gain the extra. Lots of life in a \nsimilar shape hopes to get more in life to improve. \nHall and the house would be waiting for the extra')

In [11]:
pages[0].metadata["source"]

'data_and_files\\Investment.pdf'

In [ ]:
len(pages)

In [12]:
for doc in pages:
    print("Page:", doc.metadata["page"])
    print("Source:", doc.metadata["source"])
    print("Text preview:", doc.page_content[:10], "...\n")

Page: 0
Source: data_and_files\Investment.pdf
Text preview: 1 | Page 
 ...

Page: 1
Source: data_and_files\Investment.pdf
Text preview: 2 | Page 
 ...

Page: 2
Source: data_and_files\Investment.pdf
Text preview: 3 | Page 
 ...

Page: 3
Source: data_and_files\Investment.pdf
Text preview: 4 | Page 
 ...

Page: 4
Source: data_and_files\Investment.pdf
Text preview: 5 | Page 
 ...

Page: 5
Source: data_and_files\Investment.pdf
Text preview: 6 | Page 
 ...

Page: 6
Source: data_and_files\Investment.pdf
Text preview: 7 | Page 
 ...

Page: 7
Source: data_and_files\Investment.pdf
Text preview: 8 | Page 
 ...

Page: 8
Source: data_and_files\Investment.pdf
Text preview: 9 | Page 
 ...

Page: 9
Source: data_and_files\Investment.pdf
Text preview: 10 | Page  ...

Page: 10
Source: data_and_files\Investment.pdf
Text preview: 11 | Page  ...

Page: 11
Source: data_and_files\Investment.pdf
Text preview: 12 | Page  ...

Page: 12
Source: data_and_files\Investment.pdf
Text preview: 13 | Page  ...

Page: 13


In [13]:
pages[0].metadata

{'producer': 'Microsoft® Word 2019',
 'creator': 'Microsoft® Word 2019',
 'creationdate': '2024-01-13T05:50:05+02:00',
 'author': 'net_pc',
 'moddate': '2024-01-13T05:50:05+02:00',
 'source': 'data_and_files\\Investment.pdf',
 'total_pages': 65,
 'page': 0,
 'page_label': '1'}

In [14]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs_chunks = splitter.split_documents(pages)

In [15]:
len(docs_chunks)

109

In [16]:
docs_chunks[0]

Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2024-01-13T05:50:05+02:00', 'author': 'net_pc', 'moddate': '2024-01-13T05:50:05+02:00', 'source': 'data_and_files\\Investment.pdf', 'total_pages': 65, 'page': 0, 'page_label': '1'}, page_content='1 | Page \nProf. Dr. Turgut Tursoy Investment \n \n \n \n \n \n \n \n \nInvestment \nProf. Dr. Turgut Tursoy \n \n \nIntro \n \nStart to understand that the in and waste would be \nfinding them together to create the new initiatives. It \nis the investment in life to understand a life that goes \nwith the extra efforts to gain the extra. Lots of life in a \nsimilar shape hopes to get more in life to improve. \nHall and the house would be waiting for the extra')

In [17]:
print(docs_chunks[0].page_content[:100])

1 | Page 
Prof. Dr. Turgut Tursoy Investment 
 
 
 
 
 
 
 
 
Investment 
Prof. Dr. Turgut Tursoy 
 


In [18]:
print(docs_chunks[0].metadata)

{'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2024-01-13T05:50:05+02:00', 'author': 'net_pc', 'moddate': '2024-01-13T05:50:05+02:00', 'source': 'data_and_files\\Investment.pdf', 'total_pages': 65, 'page': 0, 'page_label': '1'}


In [24]:
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = FAISS.from_documents(docs_chunks, embeddings)

In [27]:
# number of vectors 
vectorstore.index.ntotal

109

In [26]:
# vector dimension
vectorstore.index.d

384

In [28]:
# number of documents
len(vectorstore.docstore._dict)

109

In [ ]:
result = llm_ollama.ainvoke({
        "messages": [
            SYSTEM_PROMPT,
            HumanMessage(
                content="First add 3456 and 7890, then find the modulus of the result with 97."
            )
        ]
    })

In [ ]:
chat_completion = llm_chat_openai.chat.completions.create(
    messages=[
        {
            'role': 'user',
            'content': 'Say this is a test',
        }
    ],
    model='gpt-oss:20b',
)
print(chat_completion.choices[0].message.content)